<a href="https://colab.research.google.com/github/sumair789-lgtm/urdu-ocr-codesaviours-si26--Sumair-/blob/main/SI26_Week2_%5BSumair%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is week 2 notebook and this notebook contain the code to install libraries Like OpenCV Pillow
In the next step we clean our gathered pictures in new folder processed images and after that We find the gap on Tesseract by analysis of original image and Tesseract Output


In [1]:
!pip install opencv-python-headless pillow
import cv2
import numpy as np
from PIL import Image
import os
import matplotlib.pyplot as plt
print('Libraries loaded successfully!')

Libraries loaded successfully!


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import cv2
import os

def preprocess_image(image_path, save_path):
    # Load image
    img = cv2.imread(image_path)
    if img is None:
        print(f"Could not load: {image_path}")
        return None

    # Step 1: Convert to grayscale (removes color noise)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Step 2: Remove noise (gentle cleaning to protect small Urdu dots)
    denoised = cv2.fastNlMeansDenoising(gray, h=4)

    # Step 3: Adaptive Thresholding (handles shadows and uneven lighting)
    binary = cv2.adaptiveThreshold(
        denoised,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        11,
        2
    )

    # Save processed image
    cv2.imwrite(save_path, binary)
    return binary

# Create local output folder structure
os.makedirs('data/processed', exist_ok=True)
print('Preprocessing function ready!')

Preprocessing function ready!


In [4]:
import glob
import os

# Point to your actual Google Drive raw folder path
drive_raw_path = '/content/drive/MyDrive/urdu-ocr-si26/data/raw'

# Find all images in data/raw/ recursively across all subfolders
all_images = glob.glob(f'{drive_raw_path}/**/*.jpg', recursive=True)
all_images += glob.glob(f'{drive_raw_path}/**/*.png', recursive=True)
all_images += glob.glob(f'{drive_raw_path}/**/*.jpeg', recursive=True)
all_images += glob.glob(f'{drive_raw_path}/**/*.PNG', recursive=True)
all_images += glob.glob(f'{drive_raw_path}/**/*.JPG', recursive=True)

print(f'Found {len(all_images)} images to process')
processed_count = 0

# Loop through each found image path
for img_path in all_images:
    filename = os.path.basename(img_path)
    save_path = f'data/processed/processed_{filename}'

    # Call the function from Cell 2
    result = preprocess_image(img_path, save_path)

    if result is not None:
        processed_count += 1

print(f'Done! Processed {processed_count} images')
print('Check data/processed/ folder')

Found 172 images to process
Done! Processed 172 images
Check data/processed/ folder


In [5]:
import shutil

# Define where you want to save it in your Google Drive
destination_drive_path = '/content/drive/MyDrive/urdu-ocr-si26/data/processed'

# Copy the entire local folder to your Drive permanently
shutil.copytree('data/processed', destination_drive_path, dirs_exist_ok=True)
print("🎉 Folder successfully copied to your Google Drive!")

🎉 Folder successfully copied to your Google Drive!


In [6]:
# 1. Install Tesseract engine and the official Urdu language pack
!apt-get install -y tesseract-ocr tesseract-ocr-urd
!pip install pytesseract

import pytesseract
import glob
from PIL import Image
import os

# 2. Grab 5 processed images (checking local storage first, then Drive if local is empty)
test_images = list(glob.glob('data/processed/*.png')) + list(glob.glob('data/processed/*.jpg'))

if len(test_images) == 0:
    print("Pulling processed images directly from Google Drive...")
    test_images = list(glob.glob('/content/drive/MyDrive/urdu-ocr-si26/data/processed/*.png')) + list(glob.glob('/content/drive/MyDrive/urdu-ocr-si26/data/processed/*.jpg'))

# Limit to exactly 5 images for the test
test_images = test_images[:5]

print('=== Tesseract Results on Urdu Images ===\n')

for img_path in test_images:
    try:
        img = Image.open(img_path)

        # 'urd' tells Tesseract to use the Urdu language model
        result = pytesseract.image_to_string(img, lang='urd')

        print(f'Image: {os.path.basename(img_path)}')
        print(f'Tesseract output:\n{result.strip()}')
    except Exception as e:
        print(f'Error processing {os.path.basename(img_path)}: {e}')
    print('---')

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following NEW packages will be installed:
  tesseract-ocr-urd
0 upgraded, 1 newly installed, 0 to remove and 3 not upgraded.
Need to get 1,000 kB of archives.
After this operation, 1,413 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-urd all 1:4.00~git30-7274cfa-1.1 [1,000 kB]
Fetched 1,000 kB in 0s (2,417 kB/s)
Selecting previously unselected package tesseract-ocr-urd.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-urd_1%3a4.00~git30-7274cfa-1.1_all.deb ...
Unpacking tesseract-ocr-urd (1:4.00~git30-7274cfa-1.1) ...
Setting up tesseract-ocr-urd (1:4.00~git30-7274cfa-1.1) ...
=== Tesseract Results on Urdu Images ===

Image: processed_urdu_6.png
Tesseract output:
شں سس عر ڑم مل رح ۔
---
Image: pr

Image Analysis: processed_urdu_6.png
Actual Urdu Text: صحت سب سے بڑی نعمت ہے

Tesseract Output: شس سس عر ڈم مل رح -

The Gap (What went wrong):

Severe Character Misidentification: Tesseract completely failed to recognize a single correct word. It completely misread the letters, transforming the word "صحت" into "رح" and "سب" into "سس".

Segmentation Splitting: Because the cursive Nastaliq script joins letters fluidly together, Tesseract aggressively tried to cut up the connected text blocks. This caused it to hallucinate random standalone characters (like ش, ع, ڑ) that are completely absent in the actual sentence.


Image Analysis: processed_10045.png
Actual Urdu Text: اسپیکر نثار احمد کھوڑو نے ملاقاتیں کیں۔ اس موقع پر

Tesseract Output: یکر ڈار اح کھوڑو نے ملا تی نکییں۔ اس موحع ےر

The Gap (What went wrong):

Mangled Proper Names & Words: Tesseract completely mangled key nouns. It turned "اسپیکر" into "یکر", "نثار" into "ڈار", and "احمد" into "اح".

Ligature Confusion: The complex, stacked ligatures in "ملاقاتیں" and "کیں" were utterly broken apart, resulting in fractured gibberish tokens like "ملا تی" and "نکییں۔".

Character Mistranslation due to Dots (Nuqte): In the final phrase, Tesseract missed or misread critical dots, misidentifying the correct word "موقع" as "موحع" and changing the trailing "پر" to an disconnected "ےر".

Image Analysis: processed_urdu_19.png
Actual Urdu Text: ورزش صحت کے لیے ضروری ہے

Tesseract Output: / ور

The Gap (What went wrong):

Extreme Information Loss / Deletion: Tesseract failed to extract 80% of the sentence. It completely missed the words "صحت", "کے لیے", and "ضروری ہے".

Fragmented Root Recognition: It only managed to partially capture the very first two letters of the first word "ورزش" (extracting "ور"), cutting it off completely right after the non-connecting characters.

Symbol Hallucination: Due to layout confusion over the baseline of the Nastaliq script, it interpreted a text curve as a literal special character, outputting a random forward slash (/).

Image Analysis: processed_10034.png
Actual Urdu Text: انٹرویو میں بتایا کہ اگرچہ وہ سبزی خور ہیں لیکن

Tesseract Output: انظرواو ہیں تا ماکہ اگرجہ ووسین کی خر ہیں مین

The Gap (What went wrong):

Severe Word/Character Hallucination: Tesseract struggled to interpret the cursive loops, transforming "انٹرویو" into the nonsensical Arabic-looking string "انظرواو", and turning "سبزی" into "ووسین".

Contextual Substitution Failure: Instead of recognizing the actual text pattern, it hallucinated entirely different random Urdu functional words like "تا", "ماکہ", and "کی", completely changing the syntactic meaning.

Baseline Tracking & Word Crashing: Tesseract struggled heavily with the vertical stacking of Urdu characters (ligatures). For instance, it combined pieces of different words together, misinterpreting the final word "لیکن" as "مین", while breaking up "خور" into "خر".

Image Analysis: processed_1.png
Actual Urdu Text: اسکے ساتھ ملحقہ علاقے کے عوام نے بجلی و گیس

Tesseract Output: کے سا تھ موہ علا تے کے عوام نے می وکس

The Gap (What went wrong):

Ligature and Structure Disjointing: Tesseract completely broke apart the fluidly connected words. It failed to recognize the complex joined structure of "ملحقہ" and "علاقے", rewriting them as fractured pieces like "موہ" and "علا تے".

Character and Meaning Transformation: The compound subject at the end, "بجلی و گیس", was entirely misread due to the dense clustering of dots and cursive loops, getting translated into the bizarre, non-existent phrase "می وکس".

Isolated Character Fragmentation: Even where it detected basic sounds correctly, it split simple words into standalone segments, converting the smoothly written "ساتھ" into "سا تھ".

Summary:
Tesseract fails on Urdu because it is engineered for discrete, left-to-right alphabets where static letters sit neatly on a horizontal baseline (like Latin scripts). Conversely, Urdu text—specifically in the traditional Nastaliq style—is a right-to-left, highly cursive script where characters fluidly change shape depending on their position in a word.

Instead of sitting side-by-side, Urdu characters stack vertically into complex clusters called ligatures. Because Tesseract relies on finding distinct horizontal spaces to segment letters, it blindly fractures these ligatures. This structural mismatch causes it to drop entire text blocks, misinterpret diacritical dots (nuqte), hallucinate random characters, and scramble text into meaningless gibberish.